Capstone Two: Preprocessing and Training Data Development

This notebook prepares the cleaned dataset for machine learning modeling. The preprocessing workflow includes identifying categorical and numeric features, creating dummy variables, standardizing numeric variables, and splitting the dataset into training and testing sets. These steps ensure the data is properly structured and free from data leakage before model development.

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 120)

1. Load the Cleaned Dataset

The cleaned development dataset is loaded below. This dataset reflects all prior cleaning and exploratory analysis work completed earlier in the project.

In [2]:
df = pd.read_csv("us_lower48_hourly_clean.csv")

df["datetime"] = pd.to_datetime(df["datetime"])

df.head()

,datetime,demand_mw,coal_mw,gas_mw,nuclear_mw,hydro_mw,solar_mw,wind_mw,petroleum_mw,other_mw
0,2015-07-01 05:00:00+00:00,162827,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2015-07-01 06:00:00+00:00,335153,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2015-07-01 07:00:00+00:00,333837,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2015-07-01 08:00:00+00:00,398386,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2015-07-01 09:00:00+00:00,388954,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [3]:
df["year"] = df["datetime"].dt.year
df["month"] = df["datetime"].dt.month
df["day"] = df["datetime"].dt.day
df["hour"] = df["datetime"].dt.hour
df["dayofweek"] = df["datetime"].dt.dayofweek
df["is_weekend"] = df["dayofweek"].isin([5, 6]).astype(int)

df = df.drop(columns=["datetime"])

df.head()

,demand_mw,coal_mw,gas_mw,nuclear_mw,hydro_mw,solar_mw,wind_mw,petroleum_mw,other_mw,year,month,day,hour,dayofweek,is_weekend
0,162827,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2015,7,1,5,2,0
1,335153,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2015,7,1,6,2,0
2,333837,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2015,7,1,7,2,0
3,398386,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2015,7,1,8,2,0
4,388954,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2015,7,1,9,2,0


2. Initial Data Inspection

Before preprocessing, the dataset is inspected to confirm structure, column types, missing values, and overall dimensions. This ensures the dataset is ready for feature engineering and transformation.

In [4]:
df.shape

(58936, 15)

In [5]:
df.dtypes

demand_mw         int64
coal_mw         float64
gas_mw          float64
nuclear_mw      float64
hydro_mw        float64
solar_mw        float64
wind_mw         float64
petroleum_mw    float64
other_mw        float64
year              int32
month             int32
day               int32
hour              int32
dayofweek         int32
is_weekend        int64
dtype: object

In [6]:
df.isna().sum().sort_values(ascending=False).head(15)

demand_mw       0
coal_mw         0
gas_mw          0
nuclear_mw      0
hydro_mw        0
solar_mw        0
wind_mw         0
petroleum_mw    0
other_mw        0
year            0
month           0
day             0
hour            0
dayofweek       0
is_weekend      0
dtype: int64

3. Define Target Variable and Feature Matrix

The target variable (dependent variable) is separated from the feature matrix (independent variables). This separation is necessary prior to splitting the dataset.

In [7]:
target_col = "demand_mw"

X = df.drop(columns=[target_col])
y = df[target_col]

4. Identify Categorical and Numeric Features

Features are categorized into numeric and categorical types. Datetime values are first converted into useful time-based numeric features such as year, month, hour, and day of week. Only true categorical variables are one-hot encoded before modeling, while numeric variables may require scaling.

In [8]:
cat_cols = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()

print("Categorical columns:", cat_cols)
print("Number of categorical columns:", len(cat_cols))
print("Number of numeric columns:", len(num_cols))

Categorical columns: []
Number of categorical columns: 0
Number of numeric columns: 14


5. Train-Test Split

The dataset is split into training and testing subsets. This step is completed before scaling to prevent data leakage.

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

X_train.shape, X_test.shape

((47148, 14), (11788, 14))

6. Encode Categorical Features

Only true categorical variables are transformed into dummy variables using one-hot encoding. The original datetime column is not dummy encoded; instead, it was converted earlier into meaningful numeric time features. The training and testing datasets are aligned afterward to ensure consistent feature columns.

In [10]:
if len(cat_cols) > 0:
    X_train_dum = pd.get_dummies(X_train, columns=cat_cols, drop_first=True)
    X_test_dum  = pd.get_dummies(X_test, columns=cat_cols, drop_first=True)
    
    X_train_dum, X_test_dum = X_train_dum.align(X_test_dum, join="left", axis=1, fill_value=0)
else:
    X_train_dum = X_train.copy()
    X_test_dum = X_test.copy()

X_train_dum.shape, X_test_dum.shape

((47148, 14), (11788, 14))

7. Standardize Numeric Features

Numeric features are standardized using StandardScaler. The scaler is fit only on the training data and then applied to both training and testing datasets to avoid data leakage.

In [11]:
scaler = StandardScaler()

X_train_scaled = X_train_dum.copy()
X_test_scaled  = X_test_dum.copy()

num_cols_after = [c for c in X_train_scaled.columns if c in num_cols]

X_train_scaled[num_cols_after] = scaler.fit_transform(X_train_scaled[num_cols_after])
X_test_scaled[num_cols_after]  = scaler.transform(X_test_scaled[num_cols_after])

8. Verify Processed Datasets

The final training and testing datasets are reviewed to confirm:

Matching feature columns

Correct dimensions

Successful scaling

No unexpected missing values

In [12]:
print("Train X:", X_train_scaled.shape)
print("Test  X:", X_test_scaled.shape)
print("Train y:", y_train.shape)
print("Test  y:", y_test.shape)

Train X: (47148, 14)
Test  X: (11788, 14)
Train y: (47148,)
Test  y: (11788,)


In [13]:
X_train_scaled.head()

,coal_mw,gas_mw,nuclear_mw,hydro_mw,solar_mw,wind_mw,petroleum_mw,other_mw,year,month,day,hour,dayofweek,is_weekend
56245,0.621413,0.872110,0.971019,0.379571,3.286096,0.689004,0.840993,0.794280,1.344289,1.262266,1.511791,0.938254,-1.499688,-0.632920
40576,0.295018,0.737206,0.839883,1.116526,1.827763,0.421719,0.428929,0.534710,0.837115,-1.315506,-0.081241,1.371953,0.997832,1.579978
54050,1.304691,0.863843,0.913754,-0.104958,-0.542084,0.092450,-0.210110,0.953530,1.344289,0.403009,1.625579,-0.651974,-1.499688,-0.632920
52983,1.784115,2.122927,0.969031,1.578639,3.520826,0.193770,1.200939,1.621396,1.344289,0.116589,0.032547,1.227387,0.498328,-0.632920
44012,1.575947,2.128080,0.993832,1.251376,1.006539,0.663284,2.422030,1.387549,0.837115,0.116589,-0.877757,-1.519371,-0.500680,-0.632920


In [14]:
X_train_scaled.isna().sum().sort_values(ascending=False).head(10)

coal_mw         0
gas_mw          0
nuclear_mw      0
hydro_mw        0
solar_mw        0
wind_mw         0
petroleum_mw    0
other_mw        0
year            0
month           0
dtype: int64

Conclusion

In this notebook, the dataset was successfully prepared for modeling by:

Separating target and feature variables

Converting the datetime field into useful time-based numeric features

Identifying categorical and numeric features

Encoding only true categorical variables with dummy variables

Standardizing numeric features using training data only

Splitting the dataset into training and testing subsets